# Cheap Regression Prediction Model

In [ ]:
from imbalance_sa.paths import PROJECT_ROOT
import pandas as pd
import numpy as np
import datetime as dt
import plotly.graph_objects as go
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
isem_niv_pimb = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "niv" / "isem_niv_and_pimb_actual.csv",
    index_col="time_UTC", parse_dates=True,
)
# Raw export has some rows out of order and exact duplicate timestamps (Nov 2023)
isem_niv_pimb = isem_niv_pimb[~isem_niv_pimb.index.duplicated(keep="first")].sort_index()

isem_wind_generation = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "wind" / "isem_wind_actual.csv",
    index_col="time_UTC", parse_dates=True,
)

isem_niv = isem_niv_pimb[['NIV (MW)']].copy()
isem_pimb = isem_niv_pimb[['PIMB (€)']].copy()

In [ ]:
from plotly.subplots import make_subplots

start = pd.Timestamp("2026-01-01")
end = start + dt.timedelta(days=10)
niv_week = isem_niv.loc[start:end]
pimb_week = isem_pimb.loc[start:end]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=niv_week.index, y=niv_week["NIV (MW)"], name="NIV (MW)"), secondary_y=False)
fig.add_trace(go.Scatter(x=pimb_week.index, y=pimb_week["PIMB (€)"], name="PIMB (€)", line=dict(color="red")), secondary_y=True)

fig.update_yaxes(title_text="NIV (MW)", secondary_y=False)
fig.update_yaxes(title_text="PIMB (€)", secondary_y=True)
fig.update_xaxes(title_text="Date")
fig.update_layout(title_text=f"ISEM NIV and PIMB from {start.date()} to {end.date()}", legend=dict(x=0.01, y=0.99), height=400, width=1600)
fig.show()

In [ ]:
semopx_prices = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "market_prices" / "semopx_market_clearing_prices.csv",
    index_col="time_utc", parse_dates=True,
)

start = pd.Timestamp("2026-01-01")
end = start + dt.timedelta(days=7)
prices_week = semopx_prices.loc[start:end]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=prices_week.index, y=semopx_prices.da_price_eur_mwh, name="DA Price (€/MWh)"))
fig.add_trace(go.Scatter(x=prices_week.index, y=semopx_prices.ida1_price_eur_mwh, name="IDA1 Price (€/MWh)"))
fig.add_trace(go.Scatter(x=prices_week.index, y=semopx_prices.ida2_price_eur_mwh, name="IDA2 Price (€/MWh)"))
fig.add_trace(go.Scatter(x=prices_week.index, y=semopx_prices.ida3_price_eur_mwh, name="IDA3 Price (€/MWh)"))
fig.add_trace(go.Scatter(x=pimb_week.index, y=pimb_week['PIMB (€)'], name="Imbalace Price (€/MWh)"))

fig.update_yaxes(title_text="Power Price (€)")
fig.update_xaxes(title_text="Date")
fig.update_layout(title_text=f"SEMOpx market prices {start.date()} to {end.date()}", legend=dict(x=0.01, y=0.99), height=600, width=1000)
fig.show()

In [ ]:
import pandas as pd
import datetime as dt
import plotly.graph_objects as go

semopx_volumes = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "market_prices" / "semopx_market_clearing_volumes.csv",
    index_col="time_utc", parse_dates=True,
)

start = pd.Timestamp("2026-01-01")
end = start + dt.timedelta(days=7)
volumes_week = semopx_volumes.loc[start:end]

fig = go.Figure()

fig.add_trace(go.Bar(x=volumes_week.index, y=volumes_week.da_volume, name="DA Volume (MW)"))
fig.add_trace(go.Bar(x=volumes_week.index, y=volumes_week.ida1_volume, name="IDA1 Volume (MW)"))
fig.add_trace(go.Bar(x=volumes_week.index, y=volumes_week.ida2_volume, name="IDA2 Volume (MW)"))
fig.add_trace(go.Bar(x=volumes_week.index, y=volumes_week.ida3_volume, name="IDA3 Volume (MW)"))
fig.add_trace(go.Bar(x=niv_week.index, y=niv_week["NIV (MW)"], name="Imbalance Volume (MW)"))

fig.update_layout(
    barmode="stack",
    title_text=f"SEMOpx market clearing volumes {start.date()} to {end.date()}",
    xaxis_title="Date",
    yaxis_title="Traded Volume (MW)",
    legend=dict(x=0.01, y=0.99),
)

fig.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

years = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

# Percentile used to define "extreme" dynamically, per year
EXTREME_PERCENTILE = 0.99

# Fixed € tiers kept for interpretability / regulatory-style reporting alongside the dynamic measure
PIMB_TIERS = {"moderate": 200, "high": 500, "severe": 1000, "extreme": 2000}
PIMB_LOW_THRESHOLD = 0  # negative pricing event

# Flag years where skew/kurtosis look suspiciously extreme (possible data quality issue)
KURTOSIS_FLAG_THRESHOLD = 50


def summarize_df(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """
    Compute descriptive + tail-risk statistics for every column in a DataFrame.
    Returns a DataFrame indexed by column name, one row of stats per column.
    """
    if df.empty:
        return pd.DataFrame()

    rows = {}
    for col in df.columns:
        clean = df[col].dropna()
        if clean.empty:
            continue
        rows[col] = {
            "count": clean.count(),
            "mean": clean.mean(),
            "median": clean.median(),
            "std": clean.std(),
            "min": clean.min(),
            "max": clean.max(),
            "range": clean.max() - clean.min(),
            "skew": stats.skew(clean),
            "kurtosis": stats.kurtosis(clean),
            "p1": clean.quantile(0.01),
            "p5": clean.quantile(0.05),
            "p10": clean.quantile(0.10),
            "p90": clean.quantile(0.90),
            "p95": clean.quantile(0.95),
            "p99": clean.quantile(0.99),
            "iqr": clean.quantile(0.75) - clean.quantile(0.25),
            "cv": clean.std() / clean.mean() if clean.mean() != 0 else np.nan,
        }
    result = pd.DataFrame(rows).T
    result.index.name = label
    return result


def get_top_extreme_values(df: pd.DataFrame, col: str, n: int = 5) -> pd.Series:
    """Returns the n largest values in a column, for sanity-checking extreme years."""
    return df[col].dropna().nlargest(n)


all_stats = []

print("Printing annual statistics for NIV and PIMB:")
for year in years:
    print(f"\n{'='*60}\nStatistics for {year}:\n{'='*60}")

    try:
        isem_niv_year = isem_niv.loc[str(year)]
        isem_pimb_year = isem_pimb.loc[str(year)]
    except KeyError:
        print(f"  No data available for {year}, skipping.")
        continue

    if isem_niv_year.empty or isem_pimb_year.empty:
        print(f"  No data available for {year}, skipping.")
        continue

    niv_stats_df = summarize_df(isem_niv_year, f"NIV_{year}")
    pimb_stats_df = summarize_df(isem_pimb_year, f"PIMB_{year}")

    print(f"\nNIV (MW) stats by column:")
    print(niv_stats_df[["mean", "median", "std", "min", "max", "p1", "p5", "p95", "p99", "skew", "kurtosis"]])

    print(f"\nPIMB (€/MWh) stats by column:")
    print(pimb_stats_df[["mean", "median", "std", "min", "max", "p1", "p5", "p95", "p99", "skew", "kurtosis"]])

    # --- Flag suspicious skew/kurtosis for manual review ---
    for col in pimb_stats_df.index:
        k = pimb_stats_df.loc[col, "kurtosis"]
        if k > KURTOSIS_FLAG_THRESHOLD:
            top_vals = get_top_extreme_values(isem_pimb_year, col, n=5)
            print(f"\n  ⚠ [{col}] Kurtosis = {k:.1f} is unusually high — check these top values for data issues:")
            print(f"    {top_vals.to_dict()}")

    # --- Dynamic (percentile-based) extreme event counts ---
    print(f"\nExtreme Event Counts (dynamic, P{int(EXTREME_PERCENTILE*100)} threshold per year):")
    for col in isem_pimb_year.columns:
        series = isem_pimb_year[col].dropna()
        n_periods = len(series)
        if n_periods == 0:
            continue
        dynamic_threshold = series.quantile(EXTREME_PERCENTILE)
        n_dynamic_extreme = (series > dynamic_threshold).sum()
        n_neg = (series < PIMB_LOW_THRESHOLD).sum()
        print(f"  [{col}] Dynamic threshold: €{dynamic_threshold:.1f} | "
              f"Events above: {n_dynamic_extreme} ({100*n_dynamic_extreme/n_periods:.2f}%) | "
              f"Negative price events: {n_neg} ({100*n_neg/n_periods:.2f}%)")

    # --- Fixed tiered € thresholds (for interpretable, cross-year comparison) ---
    print(f"\nFixed Tier Breach Counts:")
    for col in isem_pimb_year.columns:
        series = isem_pimb_year[col].dropna()
        n_periods = len(series)
        if n_periods == 0:
            continue
        tier_counts = {
            tier: (series > threshold).sum() for tier, threshold in PIMB_TIERS.items()
        }
        tier_str = " | ".join(
            f"{tier} (>€{PIMB_TIERS[tier]}): {count} ({100*count/n_periods:.3f}%)"
            for tier, count in tier_counts.items()
        )
        print(f"  [{col}] {tier_str}")

    # --- Dynamic NIV extreme events ---
    print(f"\nNIV Extreme Event Counts (dynamic, P{int(EXTREME_PERCENTILE*100)} of |NIV| per year):")
    for col in isem_niv_year.columns:
        series = isem_niv_year[col].dropna()
        n_periods = len(series)
        if n_periods == 0:
            continue
        dynamic_threshold = series.abs().quantile(EXTREME_PERCENTILE)
        n_extreme = (series.abs() > dynamic_threshold).sum()
        print(f"  [{col}] Dynamic threshold: {dynamic_threshold:.1f} MW | "
              f"Events above: {n_extreme} ({100*n_extreme/n_periods:.2f}%)")

    # --- NIV-PIMB sensitivity (using first column of each, or specify explicitly) ---
    niv_col = isem_niv_year.columns[0]
    pimb_col = isem_pimb_year.columns[0]
    combined = pd.concat(
        [isem_niv_year[niv_col], isem_pimb_year[pimb_col]], axis=1, join="inner"
    ).dropna()
    combined.columns = ["NIV", "PIMB"]
    if len(combined) > 1:
        correlation = combined["NIV"].corr(combined["PIMB"])
        slope, intercept, r_value, p_value, std_err = stats.linregress(
            combined["NIV"], combined["PIMB"]
        )
        print(f"\nNIV-PIMB Relationship ({niv_col} vs {pimb_col}):")
        print(f"  Correlation: {correlation:.3f}")
        print(f"  Linear sensitivity: {slope:.4f} €/MWh per MW of NIV (R²={r_value**2:.3f}, p={p_value:.4f})")

    niv_stats_df["year"] = year
    pimb_stats_df["year"] = year
    all_stats.append(niv_stats_df)
    all_stats.append(pimb_stats_df)

summary_df = pd.concat(all_stats)
print("\n\n" + "="*60)
print("Consolidated summary table:")
print("="*60)
print(summary_df)